# Realidad aumentada (calibración intrínseca/extrínseca de una cámara en tiempo real a partir de una plantilla planar)

## Objetivos

* Manejar los conceptos básicos de la geometría de proyección usando un modelo de cámara lineal, mediante el desarrollo de una aplicación de realidad aumentada.
* Ilustrar los procedimientos de calibración extrínseca/intrínseca.
* Motivar las ventajas de la rectificación de planos en múltiples aplicaciones.


## Requerimientos previos

Descargamos unos vídeos y una imagen para testear los desarrollos en el notebook:

In [1]:
! wget -q https://ditec.um.es/~pedroe/calibration.mp4 -O calibration.mp4
# LINK ALTERNATIVO:
# wget -q https://univmurcia-my.sharepoint.com/:v:/g/personal/pedroe_um_es/EdBBbt414ARBhzz02874rpgBw6XlEV-d1rIEIPEro_2ygQ?download=1 -O calibration.mp4
! wget -q https://ditec.um.es/~pedroe/augm_real.mp4 -O augm_real.mp4
# LINK ALTERNATIVO:
# wget -q https://univmurcia-my.sharepoint.com/:v:/g/personal/pedroe_um_es/ET7hst6ZLf9AhaOyxUUFwdYBZKPBlQXT55W59cmCdc2w_Q?download=1 -O augm_real.mp4
! wget -q https://ditec.um.es/~pedroe/fotoplantilla.png -O fotoplantilla.png
# LINK ALTERNATIVO:
# wget -q https://univmurcia-my.sharepoint.com/:i:/g/personal/pedroe_um_es/EaBCQkz67d5NpbvH9Tcy5egB3R5dTpVegkp6H-7hxvJx6Q?download=1 -O fotoplantilla.png
! pip install mpld3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 4.4 MB/s eta 0:00:00a 0:00:01


## Imports necesarios

In [2]:
# Generales (OpenCV, numpy, matplotlib, random, math, etc.):
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import mpld3
mpld3.enable_notebook()  # Interactividad matplotlib para todo el notebook
from IPython.display import display, clear_output

# Ipywidgets:
from ipywidgets import IntSlider, FloatSlider, Checkbox, interact, fixed

# ctypes (interfaz con C)
from ctypes import Structure, pointer, POINTER, c_float, c_int, cast
from numpy.ctypeslib import load_library, ndpointer


## Calibración de la cámara

Un primer paso muy habitual (en la mayoría de ocasiones, imprescindible) para inferir información geométrica de una escena a partir de imágenes o videos de la misma es la calibración de la cámara.

En nuestro caso, usaremos para ello una serie de imágenes de un patrón planar, siguiendo el método de calibración explicado en las transparencias 52 a 59 del Tema 6.

Usaremos una plantilla tipo *tablero de ajedrez*, con 9x6 intersecciones claramente distinguibles:

In [3]:
# Definición de la plantilla:
square_size = 1.0
pattern_size = (9, 6)
pattern_points = np.zeros((np.prod(pattern_size), 3), np.float32)
pattern_points[:, :2] = np.indices(pattern_size).T.reshape(-1, 2)
pattern_points *= square_size # pattern_points = array([[0., 0., 0.],[1., 0., 0.],...,[8., 0., 0.],[0., 1., 0.],...,[8., 5., 0.]], dtype=float32)

La siguiente función lee los frames de un vídeo de entrada (hasta un máximo de `nframes`), extrae de ellos las posiciones de imagen de los 9x6=54 puntos de la plantilla, y va guardando los resultados en una lista de tuplas `(frame,points)`  que se devuelve al terminar de procesar el vídeo:

In [4]:
def process_video(videofile, nframes):

  # Puntos para el matching:
  obj_points = []
  img_points = []
  h, w = 0, 0

  # Apertura del archivo de vídeo:
  videoreader = cv2.VideoCapture(videofile)

  # Bucle principal de lectura de frames:
  i, ntempl, results = 0, 0, []
  while True:
    (grabbed, img) = videoreader.read()
    if not grabbed or i==nframes: # Sólo número de frames solicitados
      break
    else:
      imggray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
      rows, cols = imggray.shape[:2]

      # Imagen para dibujar después las esquinas detectadas:
      vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

      # Encontrar y dibujar esquinas detectadas:
      flags = (cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE +
               cv2.CALIB_CB_FAST_CHECK)
      found, corners = cv2.findChessboardCorners(imggray, pattern_size, flags=flags)
      if found:
        ntempl += 1
        term = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_COUNT, 30, 0.1)
        cv2.cornerSubPix(imggray, corners, (5, 5), (-1, -1), term)
        cv2.drawChessboardCorners(vis, pattern_size, corners, found)
        msg = "({0}/{1}) detected/input frames".format(ntempl, i+1)
        clear_output(wait=True)
        display(msg)

      results.append((vis, corners))
    # Siguiente frame:
    i += 1
  # Devolvemos todos los resultados del procesamiento frame a frame:
  return results

In [5]:
results = process_video('./calibration.mp4', 100)
print("Forma del array de la primera imagen: ", results[0][0].shape)
print("Forma del array de la primera lista de puntos: ", results[0][1].shape)

'(100/100) detected/input frames'

Forma del array de la primera imagen:  (480, 640, 3)
Forma del array de la primera lista de puntos:  (54, 1, 2)


Dibujamos (con un slider interactivo para movernos por el vídeo) los puntos extraidos:

In [6]:
def plot_image(results, framepos):
  inimg, _ = results[framepos]
  fig, axes = plt.subplots(1,1,figsize=(8.,6.))
  axes.imshow(inimg, cmap='gray')
  plt.show()

In [7]:
interact(plot_image,
         results=fixed(results),
         framepos=IntSlider(value=0,min=0,max=len(results)-1,orientation='horizontal', continuous_update=False)
         );

interactive(children=(IntSlider(value=0, continuous_update=False, description='framepos', max=99), Output()), …

Comprobamos resultados, y elaboramos lista de puntos correspondientes (_matchings_) para cada frame, éstos últimos prefijados con las coordenadas canónicas de los puntos en la plantilla (nótese que el algoritmo de extracción extrae las coordenadas con precisión [_subpixel_](https://docs.opencv.org/master/dd/d1a/group__imgproc__feature.html#ga354e0d7c86d0d9da75de9b9701a9a87e) ):

In [8]:
img_points = [elem[1] for elem in results]
obj_points = [pattern_points for elem in results]
print(img_points[0][:9]) # Imprimimos última fila (en la imagen; es la primera fila en el array) de esquinas del primer frame.
print(img_points[0].shape)
print(obj_points[0][:9]) # Idem para las coordenadas canónicas de la plantilla.
print(obj_points[0].shape)

[[[512.61707 281.3531 ]]

 [[469.484   288.59622]]

 [[425.7388  295.7415 ]]

 [[382.07486 303.1726 ]]

 [[338.07837 310.50455]]

 [[294.1519  317.75266]]

 [[250.13585 325.14743]]

 [[206.38025 332.46655]]

 [[162.56514 339.63846]]]
(54, 1, 2)
[[0. 0. 0.]
 [1. 0. 0.]
 [2. 0. 0.]
 [3. 0. 0.]
 [4. 0. 0.]
 [5. 0. 0.]
 [6. 0. 0.]
 [7. 0. 0.]
 [8. 0. 0.]]
(54, 3)


Cogemos los puntos extraídos de 30 frames aleatorios, y calculamos la matriz de cámara y los parámetros de la distorsión radial con el método `cv2.calibrateCamera()` de la OpenCV (el valor de RMS nos da una idea del error de reproyección de los puntos con la calibración empleada):

In [9]:
calibframes = 30
obj_points = obj_points[:calibframes]
img_points = random.sample(img_points, calibframes)
print("Calibrating for ({0}/{1}) randomly selected images...".format(calibframes, len(results)))
rows, cols = results[0][0].shape[:2]
rmse, camera_matrix, dist_coefs, rvecs, tvecs = cv2.calibrateCamera(obj_points, img_points, (rows, cols), None, None)
print("RMSE:", rmse)
print("Camera matrix:\n", camera_matrix)
print("Distortion coefficients: ", dist_coefs.ravel())

Calibrating for (30/100) randomly selected images...
RMSE: 0.284600033143128
Camera matrix:
 [[667.18475271   0.         317.22890986]
 [  0.         667.09583365 248.12900477]
 [  0.           0.           1.        ]]
Distortion coefficients:  [ 4.05013895e-02 -1.69820502e+00 -1.59678000e-03 -5.29427459e-04
  7.41238255e+00]


A partir de las focales $f_x$ y $f_y$ (elementos en la diagonal principal de la matriz de calibración $K_{3\times 3}$ obtenida) se pueden obtener los ángulos de _field of view_ horizontal y vertical de nuestra cámara:

In [10]:
print("fov h: {0:.1f}º, fov v: {1:.1f}º".format(
  math.degrees(2*math.atan2(cols/2, camera_matrix[0][0])),
  math.degrees(2*math.atan2(rows/2, camera_matrix[1][1]))))

fov h: 51.2º, fov v: 39.6º


# Realidad aumentada

Realizar en esta sección un ejercicio completo de realidad aumentada sobre el vídeo `augm_real.mp4` (o bien un vídeo propio de la plantilla `plantilla-L-15x15.pdf` disponible en el AV; en este último caso, obviamente habrá que calibrar previamente la cámara utilizada, utilizando un procedimiento similar al seguido en los apartados anteriores de este mismo notebook).

Para ello, seguir los pasos indicados en el apartado correspondiente (2.5) de los enunciados de prácticas también disponibles en el AV.

## Notas y referencias adicionales

#### Notas

* La ayuda de la OpenCV para todas las funciones necesarias es bastante completa, por lo que se recomienda su lectura previa de forma pausada. En particular, una buena elección de opciones para las funciones de los apartados recomendados podría ser la siguiente:

|        Función         |                Opciones                |
|------------------------|----------------------------------------|
| `cv2.threshold`        | `cv2.THRESH_BINARY + cv2.THRESH_OTSU`  |
| `cv2.findContours`     | `cv2.CHAIN_APPROX_SIMPLE`              |
| `cv2.findHomography`   | `cv2.RANSAC`                           |
| `cv2.warpPerspective`  | `borderMode=cv2.BORDER_TRANSPARENT`    |

* He aquí, pues, unos buenos _snippets_ de código para utilizar en la resolución de dichos apartados obligatorios:

  `eps, closed = 5.0, True`

  `_, outimg = cv2.threshold(inimg, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)`

  `(cnts, _) = cv2.findContours(outimg.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)`

  `cntapprox = cv2.approxPolyDP(cnt, eps, closed)`
  
  `H, mask = cv2.findHomography(srcPoints, dstPoints, cv2.RANSAC, eps)`
  
  `cv2.warpPerspective(frame, warpmat, framerectifsize, dst=framerectif, borderMode=cv2.BORDER_TRANSPARENT)`

* Ídem, a la hora de dibujar contornos:

  `outimgRGB = cv2.cvtColor(outimg, cv2.COLOR_GRAY2BGR)`
  
  `cv2.drawContours(outimgRGB, cnts, -1, (255, 0, 0), 1)`


* También podrán resultar de utilidad unas cuantas funciones de la librería `numpy`, como `np.dot`, `np.cross`, `np.hstack`, `np.linalg.norm`, y `np.linalg.inv`, entre otras.

* En cualquier caso, en el apartado de referencias están los URLs con la documentación más relevante para el desarrollo de esta práctica.

#### Referencias

* Funciones relativas a contornos en OpenCV: [aquí](https://docs.opencv.org/4.5.2/d4/d73/tutorial_py_contours_begin.html).

* Funciones de dibujo en OpenCV:
[aquí](https://docs.opencv.org/4.5.2/dc/da5/tutorial_py_drawing_functions.html).

* Funciones de calibración y 3D en OpenCV:
[aquí](https://docs.opencv.org/4.5.2/d9/db7/tutorial_py_table_of_contents_calib3d.html) y [aquí](https://docs.opencv.org/master/d9/d0c/group__calib3d.html).

* Librerías Numpy y Scipy:
[aquí](http://docs.scipy.org/doc/).